[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shashvindu/ai-teacher-notebooks/blob/main/NLP/nlp_05_bag_of_words.ipynb) - run this notebook in your browser, nothing to install.

# NLP #5: Bag of Words and N-grams

**AI Teacher** · Interview prep notebook for the video *Bag of Words and N-grams* (taught by Gonu Jha Ji with Priya, Rahul and Sneha).

Run the cells from top to bottom (`Shift + Enter`). Every output you see was produced by running this notebook. Change the examples and run them again - that is the best way to learn.

## What you will learn
1. Why text must become numbers, and what a fixed-length vector is
2. **Bag of Words from scratch** in pure Python, then with scikit-learn's `CountVectorizer`
3. Test-time behaviour: `fit` vs `transform`, and what happens to new words
4. Counts vs binary features (spam example)
5. N-grams: how bigrams rescue "not good"
6. Sparse vectors and the CSR format
7. **An end-to-end sentiment baseline** - unigrams vs bigrams

In [1]:
# One-time setup (uncomment if a library is missing):
# !pip install nltk scikit-learn pandas
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
print("Ready!")

Ready!


## 1. Key definitions

| Term | Meaning |
|---|---|
| **Corpus** | All documents we work with (e.g. all reviews) |
| **Vocabulary** | Every unique word in the corpus; each word gets one column |
| **Bag of Words (BoW)** | Represent each document by how many times each vocabulary word appears. Grammar and order are ignored - the words go into a "bag" |
| **Document-term matrix** | The table of counts: one row per document, one column per vocabulary word |
| **Feature** | One column of that matrix (a word, or an n-gram) |
| **N-gram** | A sequence of N consecutive words: unigram (1), bigram (2), trigram (3) |
| **Sparse vector** | A vector where almost every value is zero |
| **CSR** (Compressed Sparse Row) | A storage format that keeps only the non-zero (column, value) pairs |

> **Gonu Jha's story:** the market register counted each shop's items and ignored the order - *Bag of Words*.
> The minister's two notes, *"no water in the milk"* and *"no milk in the water"*, had identical counts - the order problem.
> Counting word pairs caught the trick - *n-grams*. Writing only the filled boxes of the huge register - *sparse vectors*.

## 2. Why numbers?
A machine learning model (Logistic Regression, Naive Bayes...) only understands numbers. Giving each word an
ID (good = 1, bad = 2) does not work: the model would think *bad* is twice *good*. Instead, every document becomes a
**vector of the same length** - one number per vocabulary word.

## 3. Bag of Words from scratch (pure Python)

In [2]:
reviews = ["good phone good battery",
           "bad battery",
           "good camera bad phone"]

# Step 1-2: tokenize (the text is already clean and lowercase)
tokenized = [r.split() for r in reviews]

# Step 3: vocabulary = unique words, sorted, each with a column number
vocab = sorted({w for doc in tokenized for w in doc})
print("vocabulary:", {w: i for i, w in enumerate(vocab)})

# Step 4-5: count every vocabulary word in every document
matrix = [[doc.count(w) for w in vocab] for doc in tokenized]
for r, row in zip(reviews, matrix):
    print(f"{r:25} -> {row}")

vocabulary: {'bad': 0, 'battery': 1, 'camera': 2, 'good': 3, 'phone': 4}
good phone good battery   -> [0, 1, 0, 2, 1]
bad battery               -> [1, 1, 0, 0, 0]
good camera bad phone     -> [1, 0, 1, 1, 1]


Three reviews of different lengths, but every vector has exactly **5** numbers - one per vocabulary word.

## 4. The same with scikit-learn

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
X = cv.fit_transform(reviews)      # fit = learn the vocabulary, transform = count
print(cv.get_feature_names_out())
print(X.toarray())

['bad' 'battery' 'camera' 'good' 'phone']
[[0 1 0 2 1]
 [1 1 0 0 0]
 [1 0 1 1 1]]


In [4]:
# The document-term matrix as a readable table
pd.DataFrame(X.toarray(), columns=cv.get_feature_names_out(), index=reviews)

,bad,battery,camera,good,phone
good phone good battery,0,1,0,2,1
bad battery,1,1,0,0,0
good camera bad phone,1,0,1,1,1


`CountVectorizer` also does the cleaning for you: it lowercases, and its default `token_pattern`
keeps words of **2 or more** letters (so punctuation and single letters like "I" disappear).

## 5. Test time: `transform`, never `fit` again
The vocabulary is learned **once**, on the training data. New documents are only *transformed*.

In [5]:
new = ["amazing phone, good good camera"]
print(cv.transform(new).toarray())
print("'amazing' in vocabulary?", "amazing" in cv.vocabulary_)

[[0 0 1 2 1]]
'amazing' in vocabulary? False


*amazing* - the most positive word in the review - is silently **ignored** because it was never seen in training.
This is the OOV problem from NLP #3.

What goes wrong if you call `fit_transform` on test data? The vocabulary changes, so the columns no longer
match the ones the model was trained on:

In [6]:
wrong = CountVectorizer().fit(new)
print("train columns:", list(cv.get_feature_names_out()))
print("test columns :", list(wrong.get_feature_names_out()), "<- different columns, the model would read garbage")

train columns: ['bad', 'battery', 'camera', 'good', 'phone']
test columns : ['amazing', 'camera', 'good', 'phone'] <- different columns, the model would read garbage


## 6. Counts vs binary (spam example)

In [7]:
spam = ["free free free offer", "meeting at noon"]
counts = CountVectorizer().fit(spam)
binary = CountVectorizer(binary=True).fit(spam)
print("vocabulary:", counts.get_feature_names_out())
print("counts:\n", counts.transform(spam).toarray())
print("binary:\n", binary.transform(spam).toarray())

vocabulary: ['at' 'free' 'meeting' 'noon' 'offer']
counts:
 [[0 3 0 0 1]
 [1 0 1 1 0]]
binary:
 [[0 1 0 0 1]
 [1 0 1 1 0]]


Use `binary=True` when **presence** matters more than frequency - short messages, or to stop a spammer who writes *free* twenty times from dominating the features.

## 7. Limitation: word order is lost

In [8]:
v = CountVectorizer()
M = v.fit_transform(["dog bites man", "man bites dog"])
print(v.get_feature_names_out())
print(M.toarray(), "<- identical vectors, opposite meanings")

['bites' 'dog' 'man']
[[1 1 1]
 [1 1 1]] <- identical vectors, opposite meanings


### All limitations of Bag of Words
| Limitation | Example | Fix |
|---|---|---|
| Word order is lost | *dog bites man* = *man bites dog* | n-grams |
| No meaning | *good* and *great* are unrelated columns | word embeddings (NLP #7+) |
| Huge, sparse vectors | 50,000 words → 50,000 columns | sparse storage, `max_features` |
| New words ignored | *amazing* at test time | subword models, bigger training data |
| Common words dominate | *the*, *is* have the biggest counts | TF-IDF (NLP #6) |

## 8. N-grams

In [9]:
sentence = "the movie was not good".split()
for n in (1, 2, 3):
    grams = [" ".join(sentence[i:i + n]) for i in range(len(sentence) - n + 1)]
    print(f"{n}-grams ({len(grams)}):", grams)

1-grams (5): ['the', 'movie', 'was', 'not', 'good']
2-grams (4): ['the movie', 'movie was', 'was not', 'not good']
3-grams (3): ['the movie was', 'movie was not', 'was not good']


N words give N − 1 bigrams. The bigram **not good** keeps the negation as one feature.

In [10]:
docs = ["movie was good", "movie was not good"]
for rng in [(1, 1), (1, 2)]:
    v = CountVectorizer(ngram_range=rng)
    M = v.fit_transform(docs)
    print(f"ngram_range={rng}: {len(v.get_feature_names_out())} features")
    display(pd.DataFrame(M.toarray(), columns=v.get_feature_names_out(), index=docs))

ngram_range=(1, 1): 4 features


,good,movie,not,was
movie was good,1,1,0,1
movie was not good,1,1,1,1


ngram_range=(1, 2): 8 features


,good,movie,movie was,not,not good,was,was good,was not
movie was good,1,1,1,0,0,1,1,0
movie was not good,1,1,1,1,1,1,0,1


With unigrams both reviews contain *good*. With bigrams, *was good* appears only in the first and *not good* only in the second - now a model can tell them apart.

### The cost of n-grams
Features grow very fast. Control them with:
- `max_features=N` - keep only the N most frequent features
- `min_df=k` - drop features that appear in fewer than k documents (rare n-grams)
- `ngram_range=(1, 2)` is the usual sweet spot

In [11]:
corpus = ["good phone good battery", "bad battery", "good camera bad phone",
          "battery life is great", "camera quality is poor", "phone heats up fast"]
for kwargs in [dict(ngram_range=(1, 1)), dict(ngram_range=(1, 2)), dict(ngram_range=(1, 3)),
               dict(ngram_range=(1, 3), min_df=2), dict(ngram_range=(1, 3), max_features=5)]:
    n = len(CountVectorizer(**kwargs).fit(corpus).get_feature_names_out())
    print(f"{str(kwargs):45} -> {n:3} features")

{'ngram_range': (1, 1)}                       ->  13 features
{'ngram_range': (1, 2)}                       ->  29 features
{'ngram_range': (1, 3)}                       ->  39 features
{'ngram_range': (1, 3), 'min_df': 2}          ->   6 features
{'ngram_range': (1, 3), 'max_features': 5}    ->   5 features


## 9. Sparse vectors

In [12]:
X = CountVectorizer().fit_transform(corpus)
print(type(X).__name__, X.shape, "non-zeros:", X.nnz)
print(f"sparsity: {1 - X.nnz / (X.shape[0] * X.shape[1]):.0%} of the cells are zero")
print()
print("Row 0 as stored (column, value):")
print(X[0])

csr_matrix (6, 13) non-zeros: 21
sparsity: 73% of the cells are zero

Row 0 as stored (column, value):
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3 stored elements and shape (1, 13)>
  Coords	Values
  (0, 4)	2
  (0, 9)	1
  (0, 1)	1


With a real vocabulary of 50,000 words, a 20-word tweet fills at most 20 columns: **99.96%** zeros.
A dense matrix of 1 million tweets × 50,000 words would need about 400 GB of memory;
the sparse version needs only the non-zeros. **Never call `.toarray()` on a big corpus** - scikit-learn models
like `MultinomialNB` and `LogisticRegression` accept sparse input directly.

## 10. End-to-end: a sentiment baseline (Sneha's assignment!)
Train the same Naive Bayes model twice - once with unigrams, once with unigrams + bigrams - and test it on
reviews that contain negations.

In [13]:
from sklearn.naive_bayes import MultinomialNB

train = ["good phone", "great battery", "camera is good", "love this phone", "not bad at all",
         "battery is not bad", "bad phone", "poor battery", "camera is bad", "not good at all",
         "screen is not good", "battery is not great"]
labels = [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]          # 1 = positive, 0 = negative
test = ["the camera is not bad", "good battery", "phone is not good", "not bad phone"]
truth = [1, 1, 0, 1]

for rng in [(1, 1), (1, 2)]:
    vec = CountVectorizer(ngram_range=rng)
    model = MultinomialNB().fit(vec.fit_transform(train), labels)
    pred = model.predict(vec.transform(test)).tolist()
    acc = sum(p == t for p, t in zip(pred, truth)) / len(truth)
    print(f"ngram_range={rng}: predictions={pred}  truth={truth}  accuracy={acc:.0%}")

ngram_range=(1, 1): predictions=[0, 1, 0, 1]  truth=[1, 1, 0, 1]  accuracy=75%
ngram_range=(1, 2): predictions=[1, 1, 0, 1]  truth=[1, 1, 0, 1]  accuracy=100%


With unigrams, *"the camera is not bad"* is called **negative**: it sees *not* and *bad*, both negative words. With bigrams, *not bad* is a feature of its own that the model learned as positive - so the review is classified correctly. Always start with a simple, explainable baseline like this before trying BERT.

## 11. Interview questions (from the video)

**Q1. What is Bag of Words?**
It represents text as word counts over a fixed vocabulary: build the vocabulary, then count each word in each
document. Grammar and order are ignored. *Example:* "good phone good battery" → [0, 1, 0, 2, 1] over the
vocabulary [bad, battery, camera, good, phone].

**Q2. What are the limitations of Bag of Words?**
Loses word order; no semantics (*good* vs *great*); high-dimensional sparse vectors; new words are ignored;
common words dominate. Give the fix for each: TF-IDF for common words, n-grams for order, embeddings for meaning.

**Q3. What are n-grams and why use them?**
A contiguous sequence of N words (or characters). Bigrams keep local order, so *not good* becomes one feature;
they also capture phrases like *New Delhi*. Trade-off: many more features → use `max_features` / `min_df`.

**Q4. What is a sparse vector and why does it matter?**
A vector where almost all values are zero. BoW vectors are as long as the vocabulary but have only a few
non-zeros, so they are stored in CSR format (only column + value). Never densify a big corpus; sklearn models
accept sparse input.

### Common mistakes
- `fit_transform` on test data: the vocabulary changes → always `transform`
- Saying Bag of Words understands meaning or word order
- `ngram_range=(1, 3)` on big data without `max_features`
- Calling `.toarray()` on a large corpus and running out of memory

## 12. Practice
**Task (from the video):** build the Bag of Words matrix for *"I love NLP"* and *"I love SQL"* - first by hand, then with code. Where did **I** go?

In [14]:
# Solution
docs = ["I love NLP", "I love SQL"]
default = CountVectorizer().fit(docs)
print("default     :", default.get_feature_names_out(), "<- 'i' is missing")
keep_single = CountVectorizer(token_pattern=r"(?u)\b\w+\b").fit(docs)
print("token_pattern:", keep_single.get_feature_names_out())
print(keep_single.transform(docs).toarray())

default     : ['love' 'nlp' 'sql'] <- 'i' is missing
token_pattern: ['i' 'love' 'nlp' 'sql']
[[1 1 1 0]
 [1 1 0 1]]


The default `token_pattern` keeps only tokens of 2+ characters, so the single letter *I* is dropped. Changing the pattern to `\b\w+\b` keeps it.

**Next video:** NLP #6 - TF-IDF (the story of salt and saffron).